In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    !git clone https://github.com/paulphilip-louis/attention-tracker.git /content/repo
    os.chdir("/content/repo")
    !uv pip compile pyproject.toml -o requirements.txt
    !pip install -r requirements.txt 

# Imports

In [6]:
import numpy as np
import matplotlib.pyplot
import torch as t
from utils import utils

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer

In [2]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("Loading model...")
model = utils.load_model(MODEL_NAME)


Loading model...


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B-Instruct into HookedTransformer
Moving model to device:  mps


In [3]:
normal_dataset, attack_dataset = utils.generate_dataset()

## Visualizing attention from last token to instruction

In [7]:
normal_attn_scores = utils.get_activations(model, normal_dataset[0])
injected_attn_scores = utils.get_activations(model, attack_dataset[0])

utils.plot_attn_by_layer(normal_attn_scores, injected_attn_scores)

AttributeError: module 'utils.utils' has no attribute 'plot_attn_by_layer'

## Visualizing average attention from last token to each token

In [ ]:
norm_attn_by_token = utils.get_activations_by_token(model, normal_dataset[0])
attack_attn_by_token = utils.get_activations_by_token(model, attack_dataset[0])

utils.plot_attn_by_token(norm_attn_by_token, attack_attn_by_token)

# Identifying important heads

In [10]:
scores = utils.score_heads(model, normal_dataset, attack_dataset, k=3)
utils.plot_head_scores(scores)

100%|██████████| 30/30 [00:47<00:00,  1.58s/it]


100%|██████████| 30/30 [00:18<00:00,  1.63it/s]


Computing head scores...


AttributeError: module 'utils.utils' has no attribute 'plot_head_scores'

In [11]:
H_i = utils.important_heads(scores)

# Testing performance

In [13]:
auroc, accuracy = utils.run_on_benchmark(model, H_i, threshold=0.2, benchmark_name="opi")

Loading dataset...
Building prompts
Computing focus scores for safe dataset...


  8%|▊         | 77/1000 [06:26<2:03:43,  8.04s/it]2026-04-14 12:54:04.382 python[2412:14515] Error creating directory 
 The volume ‚ÄúMacintosh HD‚Äù is out of space. You can‚Äôt save the file ‚Äúmpsgraph-2412-2026-04-14_12_53_59-418135229‚Äù because the volume ‚ÄúMacintosh HD‚Äù is out of space.
 33%|███▎      | 330/1000 [13:26<27:17,  2.44s/it]  


KeyboardInterrupt: 